# PCINet — Checkpoint Ensemble

Averages the softmax outputs of two trained checkpoints at inference time.
No retraining required.

The notebook sweeps ensemble weights from 0.1/0.9 to 0.9/0.1 and prints a results table, then shows the full confusion matrix for the best weight combination.

In [ ]:
# CONFIGURATION

# Model paths
MODEL_A = '/kaggle/input/.../best_model_run7.pt'      # Run 7 as reported in the dissertation
MODEL_B = '/kaggle/input/.../best_model_ag002.pt'     # GAIN AG w=0.02 as reported in the dissertation

# Labels used for tables
MODEL_A_LABEL = 'Run 7'
MODEL_B_LABEL = 'GAIN AG w=0.02'

# Dataset paths
DATASET_DIR = '/kaggle/input/.../mosquito_dataset'
IMAGES_DIR  = f'{DATASET_DIR}/images'
TEST_CSV    = f'{DATASET_DIR}/test.csv'
OUTPUT_DIR  = '/kaggle/working'

# Checking if the files needed from the dataset exist
for path, name in [(MODEL_A, 'MODEL_A'), (MODEL_B, 'MODEL_B'),
                    (IMAGES_DIR, 'IMAGES_DIR'), (TEST_CSV, 'TEST_CSV')]:
    import os
    print(f'{"OK" if os.path.exists(path) else "NOT FOUND"}  {name}: {path}')

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score,
    accuracy_score, cohen_kappa_score, f1_score
)
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Both checkpoints use the same EfficientNetV2-S architecture.
def build_model(checkpoint_path):
    m = models.efficientnet_v2_s(weights=None)
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(1280, 5)
    )
    m.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    return m.to(device).eval()

class CSVDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform):
        df = pd.read_csv(csv_path)
        self.imgs      = df['name'].tolist()
        self.labels    = df['label'].tolist()
        self.imgdir    = images_dir
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(os.path.join(self.imgdir, self.imgs[i])).convert('RGB')
        return self.transform(img), self.labels[i]

eval_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_loader = DataLoader(
    CSVDataset(TEST_CSV, IMAGES_DIR, eval_transforms),
    batch_size=32, shuffle=False, num_workers=2
)
print(f'Test set: {len(test_loader.dataset)} images')

# Return (N, 5) softmax probabilities and N ground-truth labels.
def get_probs(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            p = F.softmax(model(imgs.to(device)), dim=1)
            all_probs.append(p.cpu())
            all_labels.extend(labs.tolist())
    return torch.cat(all_probs, dim=0).numpy(), np.array(all_labels)

print('Setup complete.')

In [ ]:
print(f'Loading {MODEL_A_LABEL}...')
model_a = build_model(MODEL_A)
probs_a, labels = get_probs(model_a, test_loader)
del model_a  

# free GPU memory
torch.cuda.empty_cache()

print(f'Loading {MODEL_B_LABEL}...')
model_b = build_model(MODEL_B)
probs_b, _ = get_probs(model_b, test_loader)
del model_b
torch.cuda.empty_cache()

print('Inference complete.')
print(f'  probs_a shape: {probs_a.shape}  (model A softmax outputs)')
print(f'  probs_b shape: {probs_b.shape}  (model B softmax outputs)')

# Individual model results for reference
def metrics(probs, labels):
    preds = probs.argmax(axis=1)
    return {
        'MAE':      np.abs(preds - labels).mean(),
        'Soft Acc': (np.abs(preds - labels) <= 1).mean(),
        'Bal Acc':  balanced_accuracy_score(labels, preds),
        'Acc':      accuracy_score(labels, preds),
        'Kappa':    cohen_kappa_score(labels, preds),
    }

m_a = metrics(probs_a, labels)
m_b = metrics(probs_b, labels)

print(f'\nIndividual model results:')
print(f'  {MODEL_A_LABEL:<20}  MAE={m_a["MAE"]:.4f}  Soft={m_a["Soft Acc"]:.4f}  BalAcc={m_a["Bal Acc"]:.4f}')
print(f'  {MODEL_B_LABEL:<20}  MAE={m_b["MAE"]:.4f}  Soft={m_b["Soft Acc"]:.4f}  BalAcc={m_b["Bal Acc"]:.4f}')

In [ ]:
# Sweep w (weight for Model A) from 0.1 to 0.9; Model B gets (1 - w).
rows = []
weights = np.round(np.arange(0.1, 1.0, 0.1), 1)

for w in weights:
    ensemble = w * probs_a + (1 - w) * probs_b
    preds    = ensemble.argmax(axis=1)
    rows.append({
        f'w({MODEL_A_LABEL})': w,
        f'w({MODEL_B_LABEL})': round(1 - w, 1),
        'MAE':      round(np.abs(preds - labels).mean(), 4),
        'Soft Acc': round((np.abs(preds - labels) <= 1).mean(), 4),
        'Bal Acc':  round(balanced_accuracy_score(labels, preds), 4),
        'Acc':      round(accuracy_score(labels, preds), 4),
        'Kappa':    round(cohen_kappa_score(labels, preds), 4),
    })

results_df = pd.DataFrame(rows)

# Also add the individual models as reference rows at the top and bottom
ref_a = {f'w({MODEL_A_LABEL})': 1.0, f'w({MODEL_B_LABEL})': 0.0,
         'MAE': round(m_a['MAE'],4), 'Soft Acc': round(m_a['Soft Acc'],4),
         'Bal Acc': round(m_a['Bal Acc'],4), 'Acc': round(m_a['Acc'],4),
         'Kappa': round(m_a['Kappa'],4)}

ref_b = {f'w({MODEL_A_LABEL})': 0.0, f'w({MODEL_B_LABEL})': 1.0,
         'MAE': round(m_b['MAE'],4), 'Soft Acc': round(m_b['Soft Acc'],4),
         'Bal Acc': round(m_b['Bal Acc'],4), 'Acc': round(m_b['Acc'],4),
         'Kappa': round(m_b['Kappa'],4)}

full_df = pd.concat([pd.DataFrame([ref_a]), results_df, pd.DataFrame([ref_b])], ignore_index=True)

print('Ensemble weight sweep results:')
print('(w=1.0/0.0 = Model A alone, w=0.0/1.0 = Model B alone)')
print()
print(full_df.to_string(index=False))

# Find the best ensemble by MAE (lowest MAE)
best_idx = results_df['MAE'].idxmin()
best     = results_df.iloc[best_idx]
print(f'\nBest ensemble by MAE:  w({MODEL_A_LABEL})={best.iloc[0]}  '
      f'MAE={best["MAE"]}  Soft={best["Soft Acc"]}  BalAcc={best["Bal Acc"]}')

In [ ]:
# Compute confusion matrix and metrics for the best ensemble
best_w   = results_df.iloc[results_df['MAE'].idxmin()].iloc[0]
ensemble = best_w * probs_a + (1 - best_w) * probs_b
preds    = ensemble.argmax(axis=1)

cm      = confusion_matrix(labels, preds)
mae     = np.abs(preds - labels).mean()
soft    = (np.abs(preds - labels) <= 1).mean()
bal     = balanced_accuracy_score(labels, preds)
acc     = accuracy_score(labels, preds)
kappa   = cohen_kappa_score(labels, preds)

print(f'Best ensemble  w({MODEL_A_LABEL})={best_w}  w({MODEL_B_LABEL})={round(1-best_w, 1)}')
print(f'MAE={mae:.4f}  Soft Acc={soft:.4f}  Bal Acc={bal:.4f}  Acc={acc:.4f}  Kappa={kappa:.4f}')
print(f'Paper targets: MAE<0.66 {"✓" if mae < 0.66 else "✗"}  Soft Acc>0.8973 {"✓" if soft > 0.8973 else "✗"}')
print()
print('         PCI0  PCI1  PCI2  PCI3  PCI4  | recall   support')
print('-' * 62)
for i, row in enumerate(cm):
    recall = row[i] / row.sum() if row.sum() > 0 else 0
    print(f'  PCI {i}  ' + '  '.join(f'{v:4d}' for v in row) +
          f'  | {recall:.1%}    (n={row.sum()})')

In [ ]:
# Save results to CSV and confusion matrix to text file
full_df.to_csv(f'{OUTPUT_DIR}/ensemble_sweep.csv', index=False)
print(f'✓ ensemble_sweep.csv saved ({len(full_df)} rows)')

best_metrics = pd.DataFrame([{
    'mae': mae, 'soft_accuracy': soft, 'balanced_accuracy': bal,
    'accuracy': acc, 'kappa': kappa,
    f'weight_{MODEL_A_LABEL}': best_w,
    f'weight_{MODEL_B_LABEL}': round(1 - best_w, 1),
}])
best_metrics.to_csv(f'{OUTPUT_DIR}/ensemble_best_metrics.csv', index=False)
print(f'✓ ensemble_best_metrics.csv saved')

support = np.array([cm[i].sum() for i in range(5)], dtype=int)
cm_path = f'{OUTPUT_DIR}/ensemble_confusion_matrix.txt'
with open(cm_path, 'w') as f:
    f.write('=' * 70 + '\n')
    f.write('ENSEMBLE CONFUSION MATRIX\n')
    f.write(f'Model A: {MODEL_A_LABEL}  (weight={best_w})\n')
    f.write(f'Model B: {MODEL_B_LABEL}  (weight={round(1-best_w,1)})\n')
    f.write(f'Acc: {acc:.4f}  |  Bal Acc: {bal:.4f}  |  MAE: {mae:.4f}  |  Soft Acc: {soft:.4f}  |  Kappa: {kappa:.4f}\n')
    f.write('=' * 70 + '\n\n')
    f.write(f'  {"True\\Pred":<12}' + ''.join(f'  {"PCI " + str(j):<10}' for j in range(5)) + '\n')
    f.write('  ' + '-' * 60 + '\n')
    for i in range(5):
        row_str = f'  PCI {i} (n={support[i]:<4})'
        for j in range(5):
            pct   = cm[i][j] / support[i] * 100 if support[i] > 0 else 0
            count = cm[i][j]
            row_str += f'  {pct:.0f}% ({count:<3})'
        f.write(row_str + '\n')
print(f'✓ ensemble_confusion_matrix.txt saved')